# RL Programming Assignment — Prediction, Control, and Value Approximation

**Course:** ID6002W Online and Reinforcement Learning  
**Deadline:** 5th August 2026
**Total:** 100 marks  

## What students may and may not modify

Students may edit only the bodies of the required functions in cells tagged `student-todo`. Helper functions may be added inside the corresponding editable cell.

Students must not modify provided code, environments, policies, constants, imports, cell tags, function names, function signatures, or return structures. They must not add/delete/reorder cells or change the public tests. Stable-Baselines3, RLlib, CleanRL, other packaged RL solvers, global `np.random` state, action masks, and environment transition models such as `env.unwrapped.P` are prohibited.

No experiment, plotting, or figure code is required or graded.

### Required seeding convention

Every learning/evaluation function must:

1. create exactly one local generator with `rng = make_rng(seed + 1)`;
2. call `env.reset(seed=seed)` only for its first episode or rollout;
3. call `env.reset()` without a seed for subsequent episodes;
4. use only the local `rng` for policy randomness, exploration, and tie-breaking; and
5. in Question 4, call `torch.manual_seed(seed)` immediately before constructing the model.

Unless explicitly stated otherwise, `terminated=True` removes bootstrapping. `truncated=True` ends the rollout but does not remove bootstrapping. Public tests carry 0 marks and cover only basic cases.


In [1]:
# === Provided setup: do not modify ===

from __future__ import annotations

from collections.abc import Callable, Sequence

import numpy as np
import gymnasium as gym
from gymnasium import spaces

import torch
from torch import nn
import torch.nn.functional as F

BASE_SEED = 6002
DEVICE = torch.device("cpu")


def make_rng(seed: int) -> np.random.Generator:
    return np.random.default_rng(seed)

---

## Question 1 — Monte Carlo prediction in a looping MDP *(20 marks)*

`LoopWorldEnv` has states `{0,...,6}`, starts at state 3, and terminates at 0 or 6. Action 0 requests left and action 1 requests right. The requested direction is executed with probability 0.8 and reversed with probability 0.2. Entering state 0 gives −1, entering state 6 gives +1, and every other transition gives −0.02.

Evaluate the supplied policy, $\pi(\mathrm{right}\mid s)=0.65$, using $\gamma=0.95$. States stored for an episode must be the pre-action states $S_t$; `rewards[t]` is $R_{t+1}$.


In [2]:
# === Provided: LoopWorld environment and fixed policy ===


class LoopWorldEnv(gym.Env):
    metadata = {"render_modes": ["ansi"]}

    def __init__(self, max_steps: int = 200):
        super().__init__()
        self.observation_space = spaces.Discrete(7)
        self.action_space = spaces.Discrete(2)
        self.max_steps = int(max_steps)
        self.state = 3
        self.steps = 0

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        self.state = 3
        self.steps = 0
        return int(self.state), {}

    def step(self, action: int):
        if not self.action_space.contains(action):
            raise ValueError(f"invalid action {action}")
        self.steps += 1
        requested = int(action)
        executed = 1 - requested if self.np_random.random() < 0.2 else requested
        self.state = int(np.clip(self.state + (-1 if executed == 0 else 1), 0, 6))
        terminated = self.state in (0, 6)
        truncated = self.steps >= self.max_steps and not terminated
        reward = -1.0 if self.state == 0 else (1.0 if self.state == 6 else -0.02)
        info = {"requested_action": requested, "executed_action": executed}
        return self.state, reward, terminated, truncated, info

    def render(self):
        cells = ["T-", "1", "2", "3", "4", "5", "T+"]
        cells[self.state] = f"[{cells[self.state]}]"
        return " ".join(cells)


def loopworld_policy(state: int, rng: np.random.Generator) -> int:
    del state
    return int(rng.random() < 0.65)


# Supplied only for post-training evaluation and plotting.
LOOPWORLD_TRUE_VALUES = np.array(
    [0.0, -0.45592718, -0.06088703, 0.24388333, 0.51311120, 0.78165681, 0.0]
)

### Required functions and private tests *(16 marks)*


In [15]:
# === Question 1: edit only the function bodies below ===


def discounted_returns(rewards: Sequence[float], gamma: float) -> np.ndarray:
    """
    Compute the return associated with every reward index.

    Returns
    -------
    np.ndarray
        A floating-point array with shape (T,), where T == len(rewards).
        result[t] must equal rewards[t] + gamma*rewards[t+1] + ... .
        For an empty reward sequence, return an empty float array.
        Example: rewards=[1,-1,2], gamma=0.5 -> [1,0,2].
    """
    # Step 1: convert rewards to a one-dimensional float NumPy array.
    # Step 2: allocate a float result array with the same shape.
    # Step 3: scan backward while maintaining the running return G.
    # Step 4: return the result array; do not return the scalar G.
    returns = [0] * len(rewards)
    G = 0
    for i in range(len(returns)-1, -1, -1):
        G = returns[i] = rewards[i] + gamma * G 
    return np.array(returns, dtype=float)


def _generate_episode(env: gym.Env, policy: Callable[[int, np.random.Generator], int], rng: np.random.Generator):
    state_history = [env.state]    # initial state
    rewards = [0]      # for initial state, since no state transition has happened yet
    terminated = False
    while not terminated:
        action = policy(state_history[-1], rng)
        state, reward, terminated, truncated, info = env.step(action)
        state_history.append(state)
        rewards.append(reward)
        if truncated:
            return state_history, rewards, True
    return state_history, rewards, False

def mc_prediction(
    env: gym.Env,
    policy: Callable[[int, np.random.Generator], int],
    num_episodes: int,
    gamma: float,
    visit: str,
    seed: int,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Estimate V^pi by first-visit or every-visit Monte Carlo prediction.

    Returns
    -------
    (values, counts) : tuple[np.ndarray, np.ndarray]
        values: float array with shape (env.observation_space.n,).
            values[s] is the sample mean of all accepted returns for state s;
            it is 0.0 when counts[s] == 0. Terminal-state values remain 0.0.
        counts: integer array with the same shape.
            counts[s] is the number of returns used to update values[s].
            Terminal-state counts remain 0.
    """
    # Step 1: validate visit; allow only "first" or "every".
    # Step 2: initialise zero-valued float values and integer counts arrays.
    # Step 3: create rng = make_rng(seed + 1).
    # Step 4: generate each episode, storing pre-action states S_t and rewards R_{t+1}.
    # Step 5: discard the complete episode if it was truncated.
    # Step 6: compute all G_t values using discounted_returns.
    # Step 7: apply the requested visit rule and incremental sample-average update.
    # Step 8: return exactly (values, counts), in this order.
    assert visit in ("first", "every"), f"Invalid visit value {visit!r}"
    N_STATES = 6
    state_return_sums = np.zeros(N_STATES)
    counts = np.zeros(N_STATES)
    rng = make_rng(seed + 1)
    for _ in range(num_episodes):
        states, rewards, truncated = _generate_episode(env, policy, rng)
        if truncated:
            continue 
        returns = discounted_returns(rewards, gamma)
        visited_in_curr_episode = [False] * 6
        for s, G in zip(states, returns):
            # NOTE: doing [s-1] (array indexing) because s is 1-based (1..6) but array needs 0-based index
            if visit != 'first' or not visited_in_curr_episode[s-1]:
                state_return_sums[s-1] += G 
                counts[s-1] += 1
    values = np.divide(state_return_sums, counts, where=(counts != 0))
    return values, counts
    

In [16]:
mc_prediction(LoopWorldEnv(), loopworld_policy, 5, 0.95, 'first', 42)

(array([5.01894457e-310, 4.38231715e-001, 5.91991966e-001, 7.09661333e-001,
        7.99396470e-001, 8.87282274e-001]),
 array([ 0.,  1.,  5.,  9., 10.,  9.]))

### Inference text box *(4 marks; maximum 150 words)*

Explain why first-visit and every-visit Monte Carlo estimates can differ in a looping episode even though both estimate the same value function. Discuss correlated repeated visits, state-visitation frequency, and why the assignment discards truncated episodes. State one limitation of either estimator.

Submit the response in the **Question 1 inference text box**, not in the code submission.

### Answer

First-visit and Every-visit Monte Carlo value estimates can differ when the same state can appear multiple times in an episode. 
In "looping world" of this assignment, states can repeat as there's always a non-zero probability of going left or right in both available actions.
When a state repeats in an episode, First-visit MC considers only return of first state visit ignoring rest, whereas Every-visit MC considers all state visits.
This causes difference in value estimate.

*Correlated Repeated Visits* means that returns of repeated state visits in an episode are correlated to each other, as present return $G$ depends on both present reward and discounted future returns. Only Every-visit MC is affected by this, since correlated repeated visits of returns of a single state in an episode are included in final value average. First-visit MC is not affected.

*State-Visitation frequency* : First-visit MC only visits first occurrence of a state in each episode (if the state was visited in the episode). Every-visit MC visits each repeated occurrence of states in episodes.

Truncated episodes are discarded since Monte Carlo requires full episodes to calculate state returns. We cannot calculate returns for any state as some future state & reward info not available,

*Limitations*:
* First-visit MC discards useful data (of repeated state visits after first visit) and consequently has slower initial learning.
* Every-visit MC gives biased estimates as it violates independent returns assumption (within a single episode) due to correlated repeated visits.

---

## Question 2 — Off-policy TD control with Q-learning *(25 marks)*

Use Gymnasium `Taxi-v4`. There are six actions. An ordinary step gives −1, an illegal pickup/drop-off gives −10, and successful delivery gives +20. Train over all actions; do not use the action mask.


### Required functions and private tests *(20 marks)*


In [ ]:
# === Question 2: edit only the function bodies below ===


def epsilon_greedy(
    q_values: np.ndarray, epsilon: float, rng: np.random.Generator
) -> int:
    """
    Returns
    -------
    int
        One Python integer action in [0, len(q_values)-1]. With probability
        epsilon it is uniform over all actions; otherwise it is uniform over
        all actions tied for the largest q_value.
    """
    # Step 1: require one-dimensional q_values and 0 <= epsilon <= 1.
    # Step 2: use only rng for the explore/exploit decision and sampling.
    # Step 3: convert the selected NumPy integer to a Python int and return it.
    assert len(q_values.shape) == 1 and 0 <= epsilon <= 1
    # TODO


def q_learning_update(
    Q: np.ndarray,
    state: int,
    action: int,
    reward: float,
    next_state: int,
    terminated: bool,
    alpha: float,
    gamma: float,
) -> float:
    """
    Mutate one entry of Q using the Q-learning update.

    Returns
    -------
    float
        The new value of Q[state, action] after the in-place update.
        No copy of Q, target, or TD error should be returned.
    """
    # Step 1: target = reward if terminated, else reward + gamma*max(Q[next_state]).
    # Step 2: update Q[state, action] in place with step size alpha.
    # Step 3: return float(Q[state, action]).
    raise NotImplementedError("Implement q_learning_update.")


def q_learning(
    env: gym.Env,
    num_episodes: int,
    alpha: float,
    gamma: float,
    epsilon_schedule: Callable[[int], float],
    seed: int,
) -> tuple[np.ndarray, dict[str, np.ndarray]]:
    """
    Train a tabular Q-learning agent.

    Returns
    -------
    (Q, metrics) : tuple[np.ndarray, dict[str, np.ndarray]]
        Q: float array with shape
            (env.observation_space.n, env.action_space.n).
        metrics: dictionary with exactly four arrays of shape (num_episodes,):
            "return"  -> float; undiscounted reward sum per episode.
            "length"  -> int; number of steps per episode.
            "success" -> float values 0.0/1.0; 1 only when the episode
                         terminates on reward +20.
            "illegal" -> int; number of reward -10 transitions per episode.
    """
    # Step 1: infer table dimensions, initialise Q and all metric arrays to zero.
    # Step 2: create rng = make_rng(seed + 1).
    # Step 3: for each zero-indexed episode, reset with the specified seeding rule.
    # Step 4: call epsilon_schedule(episode) exactly once at episode start.
    # Step 5: interact until terminated or truncated and update Q every transition.
    # Step 6: record the four metrics using the definitions above.
    # Step 7: return exactly (Q, metrics), with no additional dictionary keys.
    raise NotImplementedError("Implement q_learning.")


def evaluate_greedy(
    env: gym.Env, Q: np.ndarray, num_episodes: int, seed: int
) -> dict[str, float]:
    """
    Evaluate Q with epsilon=0 and no learning.

    Returns
    -------
    dict[str, float]
        A dictionary with exactly:
        "mean_return"        -> mean undiscounted episode return.
        "success_rate"       -> fraction terminating on reward +20.
        "mean_length"        -> mean number of steps per episode.
        "illegal_action_rate"-> total reward -10 transitions / total steps.
        Every dictionary value must be a Python float.
    """
    # Step 1: create rng and evaluate num_episodes using the required reset rule.
    # Step 2: choose actions through epsilon_greedy(Q[state], 0.0, rng).
    # Step 3: do not modify Q.
    # Step 4: aggregate and return exactly the four floats listed above.
    raise NotImplementedError("Implement evaluate_greedy.")

In [ ]:
# === Provided: epsilon schedules ===


def constant_epsilon(value: float):
    return lambda episode: float(value)


def linear_epsilon_schedule(start: float, end: float, decay_episodes: int):
    def schedule(episode: int):
        fraction = min(max(episode, 0) / max(decay_episodes, 1), 1.0)
        return float(start + fraction * (end - start))

    return schedule

### Inference text box *(5 marks; maximum 150 words)*

Explain why Q-learning is off-policy when actions are generated by an epsilon-greedy behavior policy. Compare the expected advantages and risks of constant epsilon and a linearly decaying epsilon. Explain what the returned `success` and `illegal` metrics reveal beyond episode return, and state one limitation.

Submit the response in the **Question 2 inference text box**, not in the code submission.


---

## Question 3 — Temporal credit assignment with TD(λ) *(25 marks)*

`RandomWalk19Env` has non-terminal states 1–19, starts at state 10, and terminates at 0 or 20. The fixed policy chooses left/right uniformly. Entering 0 gives −1, entering 20 gives +1, and other transitions give zero. Use $\gamma=1$ in the required experiment.


In [ ]:
# === Provided: 19-state Random Walk and exact values ===


class RandomWalk19Env(gym.Env):
    def __init__(self):
        super().__init__()
        self.observation_space = spaces.Discrete(21)
        self.action_space = spaces.Discrete(2)
        self.state = 10

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        self.state = 10
        return self.state, {}

    def step(self, action: int):
        if not self.action_space.contains(action):
            raise ValueError(f"invalid action {action}")
        self.state += -1 if action == 0 else 1
        terminated = self.state in (0, 20)
        reward = -1.0 if self.state == 0 else (1.0 if self.state == 20 else 0.0)
        return self.state, reward, terminated, False, {}


def random_walk_policy(state: int, rng: np.random.Generator) -> int:
    del state
    return int(rng.integers(2))


RANDOM_WALK_TRUE_VALUES = np.zeros(21)
RANDOM_WALK_TRUE_VALUES[1:20] = np.arange(1, 20) / 10.0 - 1.0

### Required functions and private tests *(20 marks)*


In [ ]:
# === Question 3: edit only the function bodies below ===


def update_accumulating_trace(
    trace: np.ndarray, state: int, gamma: float, lam: float
) -> np.ndarray:
    """
    Update an accumulating eligibility trace in place.

    Returns
    -------
    np.ndarray
        The same array object supplied as trace, after first applying
        trace *= gamma*lam and then trace[state] += 1.0.
    """
    # Step 1: decay every existing trace entry in place.
    # Step 2: increment the current state after decay.
    # Step 3: return trace itself, not a separate copy or scalar.
    raise NotImplementedError("Implement update_accumulating_trace.")


def td_lambda_prediction(
    env: gym.Env,
    policy: Callable[[int, np.random.Generator], int],
    num_episodes: int,
    alpha: float,
    gamma: float,
    lam: float,
    seed: int,
) -> tuple[np.ndarray, dict[str, np.ndarray]]:
    """
    Run online backward-view TD(lambda) prediction.

    Returns
    -------
    (values, diagnostics) : tuple[np.ndarray, dict[str, np.ndarray]]
        values: float array with shape (env.observation_space.n,).
            values[s] is the estimate after all episodes. The two terminal
            entries values[0] and values[-1] must remain 0.0.
        diagnostics: dictionary with exactly one key, "value_history".
            diagnostics["value_history"] is a float array with shape
            (num_episodes, env.observation_space.n); row k contains the
            complete values array after episode k+1.
    """
    # Step 1: require 0 <= lam <= 1; otherwise raise ValueError.
    # Step 2: initialise values/history and create rng = make_rng(seed + 1).
    # Step 3: reset a zero trace at the beginning of each episode.
    # Step 4: compute delta from pre-update values; omit bootstrap only if terminated.
    # Step 5: decay then increment the trace; update all values with alpha*delta*trace.
    # Step 6: restore terminal values to zero after every update.
    # Step 7: save one history row after each episode.
    # Step 8: return exactly (values, {"value_history": value_history}).
    raise NotImplementedError("Implement td_lambda_prediction.")

### Inference text box *(5 marks; maximum 150 words)*

Explain how lambda changes temporal credit assignment and variance. Show why lambda 0 reduces to TD(0), explain the relationship between lambda 1 and Monte Carlo returns, and explain why decay-before-increment is necessary for the trace recurrence specified here. State one limitation.

Submit the response in the **Question 3 inference text box**, not in the code submission.


---

## Question 4 — Neural state-value approximation *(30 marks)*

Use `MountainCar-v0` with `max_episode_steps=500` to predict the value of the supplied fixed epsilon-soft heuristic policy. This is policy evaluation, not control. Do not alter the policy or `ValueNetwork` architecture.


In [ ]:
# === Provided: fixed policy and required network architecture ===


def mountaincar_policy(observation: np.ndarray, rng: np.random.Generator) -> int:
    if rng.random() < 0.05:
        return int(rng.integers(3))
    return 2 if float(observation[1]) >= 0.0 else 0


class ValueNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
        )

    def forward(self, states: torch.Tensor) -> torch.Tensor:
        return self.net(states).squeeze(-1)

### Required functions and private tests *(24 marks)*


In [ ]:
# === Question 4: edit only the function bodies below ===


def normalize_states(states) -> torch.Tensor:
    """
    Normalize MountainCar position and velocity without clipping.

    Returns
    -------
    torch.Tensor
        CPU float32 tensor with exactly the same shape as states.
        output[...,0] = (states[...,0] + 0.3) / 0.9
        output[...,1] = states[...,1] / 0.07
        The input object must not be modified.
    """
    # Step 1: create a CPU float32 tensor copy of states.
    # Step 2: apply the two exact transformations above; do not clip.
    # Step 3: return the tensor, not a NumPy array.
    raise NotImplementedError("Implement normalize_states.")


def semi_gradient_td_loss(
    model: nn.Module,
    states: torch.Tensor,
    rewards: torch.Tensor,
    next_states: torch.Tensor,
    terminated: torch.Tensor,
    gamma: float,
) -> torch.Tensor:
    """
    Compute the mean one-step semi-gradient TD(0) loss for a batch.

    Returns
    -------
    torch.Tensor
        A scalar (zero-dimensional) tensor equal to mean((V(states)-target)^2),
        where target = rewards + gamma*(~terminated)*V(next_states).
        The target is detached, so gradients flow only through V(states).
    """
    # Step 1: compute current predictions with gradients enabled.
    # Step 2: under torch.no_grad(), compute next predictions and targets.
    # Step 3: return F.mse_loss(current_predictions, targets).
    raise NotImplementedError("Implement semi_gradient_td_loss.")


def train_value_network(
    env: gym.Env,
    policy: Callable[[np.ndarray, np.random.Generator], int],
    num_steps: int,
    gamma: float,
    lr: float,
    seed: int,
) -> tuple[nn.Module, dict[str, np.ndarray]]:
    """
    Train ValueNetwork online for exactly num_steps transitions.

    Returns
    -------
    (model, diagnostics) : tuple[nn.Module, dict[str, np.ndarray]]
        model: the trained ValueNetwork on CPU.
        diagnostics: dictionary with exactly:
            "loss" -> float array, shape (num_steps,), one loss per update.
            "episode_return" -> float 1-D array, one undiscounted return
                                per completed episode.
            "episode_length" -> int 1-D array, one length per completed
                                episode and the same length as episode_return.
        Do not append an unfinished partial episode at the final step.
    """
    # Step 1: torch.manual_seed(seed), create local rng, model, and Adam optimizer.
    # Step 2: reset once with seed and initialise diagnostics/episode accumulators.
    # Step 3: for each of exactly num_steps transitions, choose action and step env.
    # Step 4: normalize a one-state batch and compute semi-gradient_td_loss.
    # Step 5: zero gradients, backpropagate, clip norm to 10, and optimizer.step().
    # Step 6: record loss and update the current undiscounted return/length.
    # Step 7: on termination or truncation, append episode values and reset unseeded.
    # Step 8: return exactly (model, diagnostics), with no extra keys.
    raise NotImplementedError("Implement train_value_network.")


@torch.no_grad()
def predict_values(model: nn.Module, states) -> np.ndarray:
    """
    Predict values for a batch of raw, unnormalized states.

    Returns
    -------
    np.ndarray
        A finite floating-point array with shape (n,), where n == len(states)
        and result[i] is model's V(states[i]). Preserve input order. Normalize
        internally, set model.eval(), and return CPU NumPy values.
    """
    # Step 1: set evaluation mode and normalize states internally.
    # Step 2: run the model with gradients disabled.
    # Step 3: detach, move to CPU, convert to NumPy, reshape to (n,), and return.
    raise NotImplementedError("Implement predict_values.")

### Inference text box *(6 marks; maximum 150 words)*

Explain why the one-step TD target must be detached in semi-gradient learning, why terminated and truncated transitions are treated differently, and why input normalization is useful. Briefly distinguish state-coverage error, bootstrapping error, and limited model capacity as possible sources of value-prediction error. State one limitation.

Submit the response in the **Question 4 inference text box**, not in the code submission.
